# NB99 UPDATED — Validate and Package the Complete Reproducible Project

This final notebook validates the expected outputs from NB00–NB07 and creates a single ZIP for manuscript review and reproducibility.

**Excluded intentionally:** the raw dataset in `01_DATA/`.

**Included:** results, figures, tables, logs, exports, README/config, and only the definitive notebooks used in the final workflow. Older superseded notebooks with known execution issues are not included.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, hashlib, shutil, platform, sys
from datetime import datetime, timezone
import pandas as pd

ROOT = Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1')
DATA = ROOT/'01_DATA'
NOTEBOOKS = ROOT/'02_NOTEBOOKS'
RESULTS = ROOT/'03_RESULTS'
MODELS = ROOT/'04_MODELS'
FIGURES = ROOT/'05_FIGURES'
TABLES = ROOT/'06_TABLES'
LOGS = ROOT/'07_LOGS'
EXPORTS = ROOT/'08_EXPORTS'
FINAL_ZIP = ROOT/'09_FINAL_ZIP'

for p in [NOTEBOOKS, RESULTS, MODELS, FIGURES, TABLES, LOGS, EXPORTS, FINAL_ZIP]:
    assert p.exists(), f'Missing project folder: {p}'

print('ROOT:', ROOT)


In [ ]:
# Definitive notebook set to include in the reproducibility package.
FINAL_NOTEBOOKS = [
    'NB00_Project_Setup.ipynb',
    'NB01_Data_Audit_EDA.ipynb',
    'NB02_Dimensionality_Reduction.ipynb',
    'NB03_Baseline_Models_FIXED.ipynb',
    'NB04_Hybrid_Models_FIXED.ipynb',
    'NB05_Deep_Tabular_Models_FIXED.ipynb',
    'NB06_Statistical_Comparison_Interpretability_FIXED.ipynb',
    'NB07_ROC_Learning_Curves.ipynb',
]

missing_notebooks = [n for n in FINAL_NOTEBOOKS if not (NOTEBOOKS/n).exists()]
assert not missing_notebooks, f'Missing definitive notebooks: {missing_notebooks}'
print('Definitive notebooks found:', len(FINAL_NOTEBOOKS))


In [ ]:
# Validate required outputs and exact row counts from the completed workflow.
checks = []

def add_check(name, ok, detail):
    checks.append({'check':name, 'passed':bool(ok), 'detail':str(detail)})

required_dirs = [
    'NB00_SETUP','NB01_AUDIT_EDA','NB02_DIMENSIONALITY',
    'NB03_BASELINES','NB04_HYBRIDS','NB05_DEEP_TABULAR',
    'NB06_STATS','NB07_ROC_LEARNING'
]
for name in required_dirs:
    p = RESULTS/name
    add_check(f'results_dir_{name}', p.exists() and any(p.iterdir()), p)

# NB03
p = RESULTS/'NB03_BASELINES'/'baseline_metrics_by_fold.csv'
q = RESULTS/'NB03_BASELINES'/'baseline_oof_predictions.csv'
add_check('NB03_metric_rows_60', p.exists() and len(pd.read_csv(p))==60, p)
add_check('NB03_oof_rows_48000', q.exists() and len(pd.read_csv(q))==48000, q)

# NB04
p = RESULTS/'NB04_HYBRIDS'/'hybrid_metrics_by_fold.csv'
q = RESULTS/'NB04_HYBRIDS'/'hybrid_oof_predictions.csv'
add_check('NB04_metric_rows_90', p.exists() and len(pd.read_csv(p))==90, p)
add_check('NB04_oof_rows_72000', q.exists() and len(pd.read_csv(q))==72000, q)

# NB05
p = RESULTS/'NB05_DEEP_TABULAR'/'deep_tabular_metrics_by_fold.csv'
q = RESULTS/'NB05_DEEP_TABULAR'/'deep_tabular_oof_predictions.csv'
add_check('NB05_metric_rows_30', p.exists() and len(pd.read_csv(p))==30, p)
add_check('NB05_oof_rows_24000', q.exists() and len(pd.read_csv(q))==24000, q)

# NB06
p = RESULTS/'NB06_STATS'/'all_model_metrics_by_fold.csv'
q = RESULTS/'NB06_STATS'/'all_oof_predictions.csv'
add_check('NB06_metric_rows_180', p.exists() and len(pd.read_csv(p))==180, p)
add_check('NB06_oof_rows_144000', q.exists() and len(pd.read_csv(q))==144000, q)

# NB07
p = RESULTS/'NB07_ROC_LEARNING'/'learning_curve_by_fold.csv'
q = TABLES/'NB07_ROC_LEARNING'/'roc_auc_by_model_seed.csv'
add_check('NB07_learning_curve_rows_225', p.exists() and len(pd.read_csv(p))==225, p)
add_check('NB07_roc_rows_36', q.exists() and len(pd.read_csv(q))==36, q)

# Key final tables/figures
key_files = [
    TABLES/'NB06_STATS'/'all_models_summary.csv',
    TABLES/'NB06_STATS'/'paired_bootstrap_primary_macro_f1.csv',
    TABLES/'NB06_STATS'/'mcnemar_primary_holm.csv',
    TABLES/'NB06_STATS'/'calibration_summary.csv',
    FIGURES/'NB06_STATS'/'all_models_macro_f1.png',
    FIGURES/'NB06_STATS'/'confusion_normalized_mean_LDA_XGBoost.png',
    TABLES/'NB07_ROC_LEARNING'/'roc_auc_summary.csv',
    TABLES/'NB07_ROC_LEARNING'/'learning_curve_summary.csv',
    FIGURES/'NB07_ROC_LEARNING'/'roc_multiclass_LDA_XGBoost.png',
    FIGURES/'NB07_ROC_LEARNING'/'learning_curve_comparison_test.png',
]
for f in key_files:
    add_check(f'key_file_{f.name}', f.exists() and f.stat().st_size>0, f)

validation = pd.DataFrame(checks)
validation.to_csv(EXPORTS/'final_pipeline_validation.csv', index=False)

failed = validation[~validation.passed]
display(validation)
assert failed.empty, 'FINAL VALIDATION FAILED:\n' + failed.to_string(index=False)

print('All final validation checks PASSED.')


In [ ]:
# Stage the reproducibility package.
stage = ROOT/'_ZIP_STAGE_FINAL'
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir(parents=True)

# Include generated outputs, but never the raw dataset.
for folder in [RESULTS, MODELS, FIGURES, TABLES, LOGS, EXPORTS]:
    if folder.exists():
        shutil.copytree(folder, stage/folder.name, dirs_exist_ok=True)

# Include only definitive notebooks.
nb_stage = stage/'02_NOTEBOOKS'
nb_stage.mkdir()
for name in FINAL_NOTEBOOKS:
    shutil.copy2(NOTEBOOKS/name, nb_stage/name)

# Include project-level documentation/config.
for name in ['README_PROJECT.md','project_config.json']:
    src = ROOT/name
    if src.exists():
        shutil.copy2(src, stage/name)

# Package metadata.
package_info = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'project': 'DRY_BEAN_HYBRID_Q1',
    'dataset_included': False,
    'raw_data_excluded_reason': 'Raw tabular dataset intentionally excluded from the reproducibility ZIP.',
    'definitive_notebooks': FINAL_NOTEBOOKS,
    'python_version': sys.version,
    'platform': platform.platform(),
    'validation_file': '08_EXPORTS/final_pipeline_validation.csv',
}
with open(stage/'PACKAGE_INFO.json','w',encoding='utf-8') as f:
    json.dump(package_info,f,indent=2,ensure_ascii=False)

# Create a manifest of every staged file except the manifest itself.
manifest_rows = []
for p in sorted(stage.rglob('*')):
    if p.is_file() and p.name != 'PACKAGE_MANIFEST.csv':
        sha = hashlib.sha256(p.read_bytes()).hexdigest()
        manifest_rows.append({
            'relative_path': str(p.relative_to(stage)),
            'size_bytes': p.stat().st_size,
            'sha256': sha
        })
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(stage/'PACKAGE_MANIFEST.csv',index=False)

print('Staged files:', len(manifest))
display(manifest.tail(20))


In [ ]:
# Create/replace final ZIP.
zip_base = FINAL_ZIP/'DRY_BEAN_HYBRID_Q1_COMPLETE_REPRODUCIBLE'
zip_path = Path(str(zip_base)+'.zip')
if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(str(zip_base),'zip',root_dir=stage)
shutil.rmtree(stage)

zip_sha = hashlib.sha256(zip_path.read_bytes()).hexdigest()
zip_info = {
    'zip_path': str(zip_path),
    'size_bytes': zip_path.stat().st_size,
    'size_mb': round(zip_path.stat().st_size/1024/1024,2),
    'sha256': zip_sha,
}
with open(EXPORTS/'final_zip_info.json','w',encoding='utf-8') as f:
    json.dump(zip_info,f,indent=2)

print('FINAL ZIP created:', zip_path)
print('Size (MB):', zip_info['size_mb'])
print('SHA256:', zip_sha)
print('\nNB99 UPDATED completed successfully.')


After successful execution, use:

`09_FINAL_ZIP/DRY_BEAN_HYBRID_Q1_COMPLETE_REPRODUCIBLE.zip`

This package excludes the raw dataset but includes the definitive notebooks and all generated outputs needed for manuscript review and reproducibility auditing.
